# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ray007herowars-cmyk/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The unit of analysis is one content item for one client on one report date.

I will use the `fact_content_daily_performance` warehouse table, whose grain is `report_date × client × content`.

I will work on the March 2026 partition (`month=2026-03`) as a mid-panel development window. I will keep the final June 2026 month sealed as a later test period.

The decision is which content items should be prioritized for editorial review. The eventual output will be a ranking or priority score.

I will use performance signals that are available at the decision moment as features. A later performance outcome will be treated separately as the label or proxy and will not be used as a feature.
I will use performance and content signals that are available before the decision moment as features. A future performance outcome will be treated separately as the label or proxy and will not be used as a feature.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 verification setup
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

print("Warehouse connection configured.")

Warehouse connection configured.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature
The features will be observable search and performance signals such as impressions, clicks, average position, and engagement-related metrics that are available before the decision moment.

### Label / proxy
The label should represent a later observed performance outcome. For this stage, I will keep the eventual outcome separate from the feature set rather than treating a rule-defined decline flag as a true target.

### Context
`client_id`, `content_id`, and `report_date` are context fields. They are useful for grouping, joining, validating the grain, and defining time windows, but they will not be model features.

### Excluded
I will exclude future information and any product-decision flags or derived scores. I will also exclude identifiers from the feature set because pseudonymous IDs should not be learned as predictive signals.

### Missing values
I will treat missingness as meaningful data quality information and will not blindly replace missing values with zero. Missingness may follow content type or client history.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Field classification recorded: features, label/proxy, context, and excluded.")

Field classification recorded: features, label/proxy, context, and excluded.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I will verify three things about the March 2026 warehouse slice:

1. The grain: one row should represent one content item for one client on one report date.
2. The row count and date range of the March 2026 slice.
3. Data availability using the warehouse availability flag with `IS TRUE`.

In [28]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

In [29]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM {REL}
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [31]:
count_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {REL}
""").df()

count_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [30]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE client_has_ga4 IS TRUE
    ) AS available_rows
FROM {REL}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,6822637


### Five initial features

I will build five features from the March 2026 daily performance data, aggregated to the content-item level.

1. **March impressions**: total `gsc_impressions` across March.

   * Knowable at the decision moment because these impressions have already occurred.

2. **March clicks**: total `gsc_clicks` across March.

   * Knowable at the decision moment because these clicks have already occurred.

3. **Average search position**: mean `gsc_avg_position` across March, using available GSC observations.

   * Knowable at the decision moment because the search positions are observed during the development window.

4. **March sessions**: total `ga4_sessions` across March.

   * Knowable at the decision moment because these sessions have already occurred.

5. **March engaged sessions**: total `ga4_engaged_sessions` across March.

   * Knowable at the decision moment because these engagement observations have already occurred.

I will keep identifiers and availability flags as context rather than model features.


In [33]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,

    AVG(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN gsc_avg_position
        END
    ) AS avg_search_position,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_sessions
            ELSE 0
        END
    ) AS march_sessions,

    SUM(
        CASE
            WHEN ga4_data_available IS TRUE
            THEN ga4_engaged_sessions
            ELSE 0
        END
    ) AS march_engaged_sessions

FROM {REL}
GROUP BY client_hash_id, content_hash_id
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,march_impressions,march_clicks,avg_search_position,march_sessions,march_engaged_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,3.0,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,0.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,7.0,0.0


In [34]:
availability_summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {REL}
""").df()

availability_summaryavailability_summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM {REL}
""").df()

availability_summary

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


### Deliberate leakage experiment

To demonstrate leakage, I will create a feature directly from the outcome being evaluated. This column is deliberately invalid because it contains information derived from the label.

I expect a simple score using this leaked feature to appear unrealistically strong. I will then remove the leaked column and keep only features that would genuinely be available at the decision moment.

This demonstrates why a feature can produce a very strong score while still being unusable in a real ranking system.


In [35]:
import numpy as np

leak_test = features.copy()

# Deliberate leakage: create a feature directly from one of the observed outcomes.
leak_test["leak_feature"] = (
    leak_test["march_clicks"] > leak_test["march_clicks"].median()
).astype(int)

# Deliberate target for the experiment.
leak_test["target"] = leak_test["leak_feature"]

# The leaked feature is exactly the target, so the score is intentionally perfect.
leak_score = (
    leak_test["leak_feature"] == leak_test["target"]
).mean()

print(f"Leaky quick score: {leak_score:.3f}")

Leaky quick score: 1.000


In [36]:
# Remove the deliberately leaked column before continuing.
features = features.drop(columns=["leak_feature"], errors="ignore")

print("Leaked feature removed. Honest feature set retained.")
print(features.columns.tolist())

Leaked feature removed. Honest feature set retained.
['client_hash_id', 'content_hash_id', 'march_impressions', 'march_clicks', 'avg_search_position', 'march_sessions', 'march_engaged_sessions']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The warehouse has uneven history across clients, so March 2026 does not necessarily represent the same amount of historical information for every client.

GA4 data is also not available for every row. Rows before a client's `ga4_data_start` can contain zero-filled GA4 values while `ga4_data_available` is FALSE, so those zeros cannot automatically be interpreted as real zero engagement.

The March 2026 development slice is also only one month. It is useful for developing the feature logic, but it does not establish that the resulting ranking will generalize to every client or future period.

A further limitation is that this data can support measured and directional decision-support claims, but it cannot by itself prove that refreshing a page caused its later performance to improve.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Limitation recorded: client history and data availability are uneven across the warehouse.")

Limitation recorded: client history and data availability are uneven across the warehouse.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.